# Demo micro-experiment

This notebook is a descriptive integration test of the Phase 2 pipeline. It
generates 100 independent base graphs `G0` with `(h1, p1) = (0.5, 0.2)` and grows
each one into five phase-2 arms that span the full homophily grid
`h2 in {0.0, 0.2, 0.5, 0.8, 1.0}` with `p2 = p1 = 0.2`; the baseline is
`(0.5, 0.2)`. Cascades use `phi = 0.15`, the second point of the study grid in
`docs/model.md`.

The five arms extend the `DEMO` configuration locally. `config.py` is unchanged:
the notebook keeps `DEMO` for the phase-1 parameters, sizes, and threshold, and
replaces its phase-2 arm list with the full homophily grid.

The notebook uses the public functions directly. It does not call the runner and
does not write a run. It is a demonstration outside the study design, not a
Phase 3 analysis.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

from bah_cascade.cascade import simulate_all_seeds
from bah_cascade.config import DEMO, H_VALUES
from bah_cascade.growth import GROUP_A, Network, grow_phase1, grow_phase2
from bah_cascade.metrics import connected_components, network_metrics
from bah_cascade.rng import base_seed, intervention_seed, make_rng

MASTER_SEED = 20261108
LAYOUT_SEED = 20261108

H1 = DEMO.h_values[0]
P1 = DEMO.p_values[0]
M = DEMO.network.m_values[0]
N1 = DEMO.network.n1
N2 = DEMO.network.n2
N = N1 + N2
PHI = DEMO.cascade.phi_values[0]

# The notebook extends DEMO locally to every homophily value with p2 = p1.
H2_VALUES = H_VALUES
ARMS = tuple((h2, P1) for h2 in H2_VALUES)
ARM_LABELS = ["baseline" if arm == (H1, P1) else f"h={arm[0]:g}" for arm in ARMS]

{
    "phase1": {"h1": H1, "p1": P1},
    "phase2_h2": list(H2_VALUES),
    "phase2_p2": P1,
    "m": M,
    "N1": N1,
    "N2": N2,
    "N": N,
    "replicates": DEMO.replicates,
    "phi": PHI,
    "arms": len(ARMS),
    "cascades_per_arm": DEMO.replicates * N,
}

## 1. Generate the 100 base graphs

In [ ]:
def build_base_graphs() -> list[dict[str, object]]:
    replicates: list[dict[str, object]] = []
    for replicate in range(1, DEMO.replicates + 1):
        seed = base_seed(MASTER_SEED, M, H1, P1, replicate)
        g0 = grow_phase1(M, H1, P1, N1, make_rng(seed))
        replicates.append({"replicate": replicate, "base_seed": seed, "g0": g0})
    return replicates


base_graphs = build_base_graphs()
len(base_graphs), base_graphs[0]["g0"].n_nodes

## 2. Grow the five phase-2 arms from each base graph

In [ ]:
def grow_arms(base_graphs: list[dict[str, object]]) -> None:
    for item in base_graphs:
        networks: dict[tuple[float, float], Network] = {}
        intervention_seeds: dict[tuple[float, float], int] = {}
        for h2, p2 in ARMS:
            seed = intervention_seed(MASTER_SEED, M, H1, P1, int(item["replicate"]), h2, p2)
            networks[(h2, p2)] = grow_phase2(item["g0"], M, h2, p2, N2, make_rng(seed))
            intervention_seeds[(h2, p2)] = seed
        item["networks"] = networks
        item["intervention_seeds"] = intervention_seeds


grow_arms(base_graphs)
sum(len(item["networks"]) for item in base_graphs)

## 3. Run one cascade per node on every network

In [ ]:
def seed_component_sizes(network: Network) -> np.ndarray:
    sizes = np.zeros(network.n_nodes, dtype=np.int64)
    for component in connected_components(network):
        sizes[list(component)] = len(component)
    return sizes


def collect_cascades(base_graphs: list[dict[str, object]]) -> pd.DataFrame:
    records: list[dict[str, object]] = []
    for item in base_graphs:
        for arm_index, arm in enumerate(ARMS):
            network = item["networks"][arm]
            degrees = network.degrees()
            sizes = seed_component_sizes(network)
            for result in simulate_all_seeds(network, PHI):
                records.append(
                    {
                        "replicate": item["replicate"],
                        "arm": arm_index,
                        "arm_label": ARM_LABELS[arm_index],
                        "seed_id": result.seed_id,
                        "seed_group": "A"
                        if int(network.groups[result.seed_id]) == GROUP_A
                        else "B",
                        "seed_degree": int(degrees[result.seed_id]),
                        "seed_component_size": int(sizes[result.seed_id]),
                        "S_N": result.S_N,
                        "S_A": result.S_A,
                        "S_B": result.S_B,
                        "t_s": result.t_s,
                        "is_global": int(result.is_global),
                    }
                )
    return pd.DataFrame(records)


cascades = collect_cascades(base_graphs)
cascades.shape

## 4. Representative network layout

The representative `G0` is the replicate whose baseline mean `S_N` is closest to
the median baseline mean across the 100 replicates. The five panels are its five
phase-2 arms. Each panel uses its own `spring_layout` coordinates with a
deterministic seed, so no fixed layout is shared across the arms. Node color
encodes the group; node size is proportional to degree on a scale shared by the
five panels.

In [ ]:
def to_networkx(network: Network) -> nx.Graph:
    graph = nx.Graph()
    graph.add_nodes_from(range(network.n_nodes))
    for node in range(network.n_nodes):
        for neighbor in network.neighbors(node):
            if node < int(neighbor):
                graph.add_edge(node, int(neighbor))
    return graph


baseline_arm = ARM_LABELS.index("baseline")
baseline_means = cascades.loc[cascades["arm"] == baseline_arm].groupby("replicate")["S_N"].mean()
representative_replicate = int((baseline_means - baseline_means.median()).abs().idxmin())
representative = base_graphs[representative_replicate - 1]
representative_networks = [representative["networks"][arm] for arm in ARMS]
representative_graphs = [to_networkx(network) for network in representative_networks]
positions = [
    nx.spring_layout(graph, seed=LAYOUT_SEED + index, method="force")
    for index, graph in enumerate(representative_graphs)
]

degrees = [network.degrees() for network in representative_networks]
degree_min = min(int(values.min()) for values in degrees)
degree_max = max(int(values.max()) for values in degrees)

fig, axes = plt.subplots(1, len(ARMS), figsize=(4.2 * len(ARMS), 5.5))
for ax, graph, network, label, arm, degree, panel_positions in zip(
    axes,
    representative_graphs,
    representative_networks,
    ARM_LABELS,
    ARMS,
    degrees,
    positions,
    strict=True,
):
    colors = np.where(network.groups == GROUP_A, "#1f77b4", "#ff7f0e")
    sizes = 25 + 175 * (degree - degree_min) / max(degree_max - degree_min, 1)
    nx.draw_networkx_edges(graph, panel_positions, ax=ax, alpha=0.28, width=0.7)
    nx.draw_networkx_nodes(
        graph,
        panel_positions,
        ax=ax,
        node_color=colors,
        node_size=sizes,
        linewidths=0,
    )
    title = (
        f"baseline (h={arm[0]:g}, p={arm[1]:g})"
        if label == "baseline"
        else f"{label} (p={arm[1]:g})"
    )
    ax.set_title(title)
    ax.set_axis_off()

legend = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#1f77b4", markersize=8, label="A"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#ff7f0e", markersize=8, label="B"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, frameon=False)
fig.suptitle(
    f"Representative G0 (replicate {representative_replicate}), per-panel spring "
    f"layout and shared degree scale, phi={PHI:g}"
)
fig.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

## 5. Cascade-size distributions

A 2x5 grid of 10-bin step histograms over the 500 nodes, split by arm: each
column is one arm. The top row pools all 100 `G0` (50,000 cascades per arm); the
bottom row shows the representative `G0` (500 cascades per arm). The vertical
axis is the fraction of cascades per bin, so the panels remain comparable despite
the different sample sizes. The histograms replace the ECDF: they expose the
local and near-global modes directly.

In [ ]:
bins = np.linspace(0, N, 11)
subsets = [
    (
        cascades,
        f"All G0\n({DEMO.replicates * N:,} per arm)",
    ),
    (
        cascades.loc[cascades["replicate"] == representative_replicate],
        f"Replicate {representative_replicate}\n({N} per arm)",
    ),
]

fig, axes = plt.subplots(
    2,
    len(ARM_LABELS),
    figsize=(16, 8),
    sharex=True,
    sharey=True,
)
for row_index, (subset, row_title) in enumerate(subsets):
    axes[row_index, 0].set_ylabel(row_title)
    for arm_index, label in enumerate(ARM_LABELS):
        ax = axes[row_index, arm_index]
        values = subset.loc[subset["arm"] == arm_index, "S_N"].to_numpy()
        weights = np.ones(values.size) / values.size
        ax.hist(
            values,
            bins=bins,
            weights=weights,
            histtype="step",
            linewidth=1.8,
            color=f"C{arm_index}",
        )
        ax.grid(alpha=0.25)
        if row_index == 0:
            ax.set_title(label)
        if row_index == len(subsets) - 1:
            ax.set_xlabel("S_N")

fig.supylabel("fraction of cascades per bin")
fig.suptitle(f"Cascade-size distribution by arm (phi={PHI:g})")
fig.tight_layout()
plt.show()

## 6. Cascade-time distributions

A 2x5 grid of step histograms of `t_s` for global cascades, with the same
structure as the size histograms: each column is one arm, the top row pools all
100 `G0` (50,000 cascades per arm before filtering), and the bottom row shows
the representative `G0` (500 cascades per arm before filtering). Only cascades
with `is_global = 1` are used, so each panel is normalized to the fraction of
global cascades per bin. `t_s` is a non-negative integer, so the bins are
centered on integer values and shared across panels.

In [ ]:
global_times = cascades.loc[cascades["is_global"] == 1, "t_s"]
t_max = int(global_times.max())
t_bins = np.arange(-0.5, t_max + 1.5, 1.0)

time_row_titles = [
    "All G0\n(global cascades only)",
    f"Replicate {representative_replicate}\n(global cascades only)",
]

fig, axes = plt.subplots(
    2,
    len(ARM_LABELS),
    figsize=(16, 8),
    sharex=True,
    sharey=True,
)
for row_index, (subset, _) in enumerate(subsets):
    axes[row_index, 0].set_ylabel(time_row_titles[row_index])
    for arm_index, label in enumerate(ARM_LABELS):
        ax = axes[row_index, arm_index]
        ax.grid(alpha=0.25)
        if row_index == 0:
            ax.set_title(label)
        if row_index == len(subsets) - 1:
            ax.set_xlabel("t_s")

        values = subset.loc[
            (subset["arm"] == arm_index) & (subset["is_global"] == 1),
            "t_s",
        ].to_numpy()
        if values.size == 0:
            ax.text(
                0.5,
                0.5,
                "no global cascades",
                transform=ax.transAxes,
                ha="center",
                va="center",
            )
            continue

        weights = np.ones(values.size) / values.size
        ax.hist(
            values,
            bins=t_bins,
            weights=weights,
            histtype="step",
            linewidth=1.8,
            color=f"C{arm_index}",
        )

fig.supylabel("fraction of global cascades per bin")
fig.suptitle(f"Global-cascade-time distribution by arm (phi={PHI:g})")
fig.tight_layout()
plt.show()

## 7. Global cascade probability and conditional cascade time

Each line is one `G0`. The arms are ordered by `h2`, so the baseline sits in the
center. The left panel shows the per-network probability of a global cascade, the
fraction of the 500 seeds with `S_N > N/2`. The right panel shows the per-network
mean `t_s` computed only over the cascades that are global. A network with no
global cascade has no defined conditional mean, so its line is broken at that
arm. The black line is the mean across `G0` over the networks where the quantity
is defined.

In [ ]:
global_cascades = cascades.loc[cascades["is_global"] == 1]
network_summary = (
    cascades.groupby(["replicate", "arm"], as_index=False)["is_global"]
    .mean()
    .rename(columns={"is_global": "global_probability"})
)
conditional_t_s = (
    global_cascades.groupby(["replicate", "arm"], as_index=False)["t_s"]
    .mean()
    .rename(columns={"t_s": "t_s_global_mean"})
)
network_summary = network_summary.merge(conditional_t_s, on=["replicate", "arm"], how="left")

probability_pivot = network_summary.pivot(
    index="replicate", columns="arm", values="global_probability"
).reindex(columns=range(len(ARMS)))
t_s_pivot = network_summary.pivot(
    index="replicate", columns="arm", values="t_s_global_mean"
).reindex(columns=range(len(ARMS)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
for ax, pivot, title, ylabel in zip(
    axes,
    [probability_pivot, t_s_pivot],
    [
        "P(global cascade) per G0",
        "Mean t_s | global cascade per G0",
    ],
    ["P(global cascade)", "mean t_s (global cascades)"],
    strict=True,
):
    for _, row in pivot.iterrows():
        ax.plot(
            range(len(ARMS)),
            row.to_numpy(),
            color="gray",
            alpha=0.45,
            marker="o",
            markersize=4,
        )
    ax.plot(
        range(len(ARMS)),
        pivot.mean(axis=0, skipna=True).to_numpy(),
        color="black",
        linewidth=2,
        marker="o",
        label="mean across G0",
    )
    ax.set_xticks(range(len(ARMS)), ARM_LABELS)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.25)

axes[0].set_ylim(-0.02, 1.02)
axes[0].legend(frameon=False)
axes[1].set_ylim(bottom=0)
fig.suptitle("Each line is one G0 across the five phase-2 arms")
fig.tight_layout()
plt.show()

## 8. Summary table

The table has one row per arm and one column per metric. Cascade columns pool
all cascades in each arm: `100 G0 x 500 seeds = 50,000` cascades per arm. The
conditional columns `P(global | A)` and `P(global | B)` restrict the pooled
cascades to seeds in group `A` or `B`, respectively. Topology columns average
the network-level metrics from `docs/model.md` over the 100 final networks of each
arm, so comparing rows shows how phase-2 homophily reshapes the graph.

In [ ]:
summary_rows: list[dict[str, float | str]] = []
for arm_index, label in enumerate(ARM_LABELS):
    group = cascades.loc[cascades["arm"] == arm_index]
    q1, q3 = np.percentile(group["S_N"], [25, 75])
    topology = [network_metrics(item["networks"][ARMS[arm_index]]) for item in base_graphs]
    summary_rows.append(
        {
            "arm": label,
            "mean S_N": group["S_N"].mean(),
            "median S_N": group["S_N"].median(),
            "IQR S_N": q3 - q1,
            "global cascade fraction": group["is_global"].mean(),
            "P(global | A)": group.loc[group["seed_group"] == "A", "is_global"].mean(),
            "P(global | B)": group.loc[group["seed_group"] == "B", "is_global"].mean(),
            "E": np.mean([metric.n_edges for metric in topology]),
            "k_mean": np.mean([metric.k_mean for metric in topology]),
            "k_max": np.mean([metric.k_max for metric in topology]),
            "cross-edge fraction": np.mean(
                [metric.edges_cross / metric.n_edges for metric in topology]
            ),
            "n_components": np.mean([metric.n_components for metric in topology]),
            "largest component": np.mean([metric.largest_component_size for metric in topology]),
        }
    )

summary = pd.DataFrame(summary_rows).set_index("arm")
summary.index.name = "arm"
summary.round(3)